# MuJoCo Colab Test
구글 코랩에서 MuJoCo 환경이 정상적으로 렌더링 되는지 테스트하는 노트북입니다.

In [1]:
# 필수 패키지 설치
!pip install mujoco mediapy

In [4]:
import os
# 코랩(화면 없는 환경) 렌더링을 위한 필수 설정
os.environ['MUJOCO_GL'] = 'egl'

import mujoco
import mediapy as media
import time
import math

# 3개의 관절(Joint)과 모터(Actuator)가 달린 간단한 로봇 팔 모델
xml_string = """
<mujoco>
  <option gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 1 2" dir="0 -1 -1" diffuse="1 1 1"/>
    <geom type="plane" size="2 2 0.1" rgba="0.8 0.9 0.9 1"/>

    <camera name="main_cam" pos="1.5 -2.0 1.5" xyaxes="1 1 0 -1 1 2" mode="fixed"/>

    <body name="base" pos="0 0 0.1">
      <geom type="cylinder" size="0.15 0.05" rgba="0.3 0.3 0.3 1"/>

      <body name="link1" pos="0 0 0.05">
        <joint name="joint_base" type="hinge" axis="0 0 1"/>
        <geom type="capsule" size="0.06 0.3" pos="0 0 0.3" rgba="0.8 0.2 0.2 1"/>

        <body name="link2" pos="0 0 0.6">
          <joint name="joint_shoulder" type="hinge" axis="0 1 0"/>
          <geom type="capsule" size="0.05 0.25" pos="0 0 0.25" rgba="0.2 0.8 0.2 1"/>

          <body name="link3" pos="0 0 0.5">
            <joint name="joint_elbow" type="hinge" axis="0 1 0"/>
            <geom type="capsule" size="0.04 0.15" pos="0 0 0.15" rgba="0.2 0.2 0.8 1"/>
          </body>
        </body>
      </body>
    </body>
  </worldbody>

  <actuator>
    <position joint="joint_base" name="pos_base" kp="50"/>
    <position joint="joint_shoulder" name="pos_shoulder" kp="50"/>
    <position joint="joint_elbow" name="pos_elbow" kp="50"/>
  </actuator>
</mujoco>
"""

print("로봇 모델 로딩 중...")
model = mujoco.MjModel.from_xml_string(xml_string)
data = mujoco.MjData(model)
renderer = mujoco.Renderer(model, height=480, width=640)

frames = []
fps = 60
duration = 4.0  # 4초 동안 시뮬레이션

print("로봇 팔 제어 및 렌더링 진행 중...")
for i in range(int(fps * duration)):
    # 현재 시뮬레이션 시간
    t = data.time

    # math.sin() 함수를 사용해 모터에 부드러운 곡선 형태의 각도 명령(제어 신호)을 내림
    data.ctrl[0] = math.sin(t * 2.0) * 1.5        # 밑동 좌우 회전
    data.ctrl[1] = math.sin(t * 1.5) * 1.0 - 0.5  # 어깨 관절 움직임
    data.ctrl[2] = math.sin(t * 3.0) * 1.0        # 팔꿈치 관절 움직임

    # 1스텝 물리 연산 진행
    mujoco.mj_step(model, data)

    # 카메라 뷰를 지정하여 렌더링
    renderer.update_scene(data, camera="main_cam")
    frames.append(renderer.render())

print("비디오 생성 완료!")
media.show_video(frames, fps=fps)

로봇 모델 로딩 중...
로봇 팔 제어 및 렌더링 진행 중...
비디오 생성 완료!


This browser does not support the video tag.
